# Whisper's transcription plus Pyannote's Diarization

Andrej Karpathy [suggested](https://twitter.com/karpathy/status/1574476200801538048?s=20&t=s5IMMXOYjBI6-91dib6w8g) training a classifier on top of  OpenAI [Whisper](https://openai.com/blog/whisper/) model features to identify the speaker, so we can visualize the speaker in the transcript. But, as [pointed out](https://twitter.com/tarantulae/status/1574493613362388992?s=20&t=s5IMMXOYjBI6-91dib6w8g) by Christian Perone, it seems that features from whisper wouldn't be that great for speaker recognition as its main objective is basically to ignore speaker differences.

In the following, I use [**`pyannote-audio`**](https://github.com/pyannote/pyannote-audio), a speaker diarization toolkit by Hervé Bredin, to identify the speakers, and then match it with the transcriptions of Whispr. I do it on the first 30 minutes of  Lex's 2nd [interview](https://youtu.be/SGzMElJ11Cc) with Yann LeCun. Check the result [**here**](https://majdoddin.github.io/lexicap.html).

To make it easier to match the transcriptions to diarizations by speaker change, Sarah Kaiser [suggested](https://github.com/openai/whisper/discussions/264#discussioncomment-3825375) runnnig the pyannote.audio first and  then just running whisper on the split-by-speaker chunks.
For sake of performance (and transcription quality?), we attach the audio segements into a single audio file with a silent spacer as a seperator, and run whisper on it. Enjoy it!

# Preparing the audio file

 Installing `yt-dlp` and downloading the [video](https://).

In [1]:
!pip install -U yt-dlp

In [2]:
!wget -O - -q  https://github.com/yt-dlp/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz | xz -qdc| tar -x

'wget' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
!yt-dlp -xv --ffmpeg-location ffmpeg-master-latest-linux64-gpl/bin --audio-format wav  -o lecun.wav -- https://youtu.be/SGzMElJ11Cc

^C


[youtube] Extracting URL: https://youtu.be/SGzMElJ11Cc
[youtube] SGzMElJ11Cc: Downloading webpage
[youtube] SGzMElJ11Cc: Downloading tv client config
[youtube] SGzMElJ11Cc: Downloading player 9599b765-main
[youtube] SGzMElJ11Cc: Downloading tv player API JSON
[youtube] SGzMElJ11Cc: Downloading ios player API JSON
[youtube] SGzMElJ11Cc: Downloading m3u8 information
[info] SGzMElJ11Cc: Downloading 1 format(s): 251
[download] Destination: lecun.webm

[download]   0.0% of  138.37MiB at  147.93KiB/s ETA 15:57
[download]   0.0% of  138.37MiB at  443.78KiB/s ETA 05:19
[download]   0.0% of  138.37MiB at    1.01MiB/s ETA 02:16
[download]   0.0% of  138.37MiB at    1.15MiB/s ETA 02:00
[download]   0.0% of  138.37MiB at  802.95KiB/s ETA 02:56
[download]   0.0% of  138.37MiB at  821.08KiB/s ETA 02:52
[download]   0.1% of  138.37MiB at    1.26MiB/s ETA 01:49
[download]   0.2% of  138.37MiB at    1.90MiB/s ETA 01:12
[download]   0.4% of  138.37MiB at    2.72MiB/s ETA 00:50
[download]   0.7% of  138.

[debug] Command-line config: ['-xv', '--ffmpeg-location', 'ffmpeg-master-latest-linux64-gpl/bin', '--audio-format', 'wav', '-o', 'lecun.wav', '--', 'https://youtu.be/SGzMElJ11Cc']
[debug] Encodings: locale cp1252, fs utf-8, pref cp1252, out utf-8 (No VT), error utf-8 (No VT), screen utf-8 (No VT)
[debug] yt-dlp version stable@2025.03.31 from yt-dlp/yt-dlp [5e457af57] (pip)
[debug] Python 3.11.9 (CPython AMD64 64bit) - Windows-10-10.0.19045-SP0 (OpenSSL 3.0.13 30 Jan 2024)
[debug] exe versions: none
[debug] Optional libraries: certifi-2024.08.30, pycrypto-3.20.0, requests-2.28.2 (unsupported), sqlite3-3.45.1, urllib3-1.26.20
[debug] Proxy map: {}
[debug] Request Handlers: urllib
[debug] Plugin directories: none
[debug] Loaded 1850 extractors
[debug] [youtube] Decrypted nsig MDlGKGRK4pgGDjI => SUoaTwqPdg5e_A
[debug] Saving youtube-nsig.9599b765-main to cache
[debug] [youtube] Decrypted nsig vDMpWckirBK5buI => 1UPmiwNrNCRCOA
[debug] [youtube] SGzMElJ11Cc: ios client https formats require 



> Indented block


Cutting the first 20 minutes of the video for further process.


In [4]:
pip install pydub

Note: you may need to restart the kernel to use updated packages.


In [6]:
from pydub import AudioSegment

t1 = 0 * 1000 #Works in milliseconds
t2 = 94 * 60 * 1000

newAudio = AudioSegment.from_wav("movie.wav")
a = newAudio[t1:t2]



`pyannote.audio` seems to miss the first 0.5 seconds of the audio, and, therefore, we prepend a spcacer.

> Indented block\



In [7]:
audio = AudioSegment.from_wav("lecun1.wav")
spacermilli = 2000
spacer = AudioSegment.silent(duration=spacermilli)
audio = spacer.append(audio, crossfade=0)

audio.export('audio.wav', format='wav')

<_io.BufferedRandom name='audio.wav'>

# Pyannote's Diarization

[`pyannote.audio`](https://github.com/pyannote/pyannote-audio) is an open-source toolkit written in Python for **speaker diarization**.

Based on [`PyTorch`](https://pytorch.org) machine learning framework, it provides a set of trainable end-to-end neural building blocks that can be combined and jointly optimized to build speaker diarization pipelines.

`pyannote.audio` also comes with pretrained [models](https://huggingface.co/models?other=pyannote-audio-model) and [pipelines](https://huggingface.co/models?other=pyannote-audio-pipeline) covering a wide range of domains for voice activity detection, speaker segmentation, overlapped speech detection, speaker embedding reaching state-of-the-art performance for most of them.

Installing Pyannote and running it on the video to generate the diarizations.

In [8]:
pip install  pyannote.audio

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained('pyannote/speaker-diarization', use_auth_token=)

C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\inspect.py:988: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  if ismodule(module) and hasattr(module, '__file__'):
Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.5.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint C:\Users\UsEr\.cache\torch\pyannote\models--pyannote--segmentation\snapshots\c4c8ceafcbb3a7a280c2d357aee9fbc9b0be7f9b\pytorch_model.bin`


Model was trained with pyannote.audio 0.0.1, yours is 3.3.2. Bad things might happen unless you revert pyannote.audio to 0.x.
Model was trained with torch 1.10.0+cu102, yours is 2.6.0+cu118. Bad things might happen unless you revert torch to 1.x.


C:\Users\UsEr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\speechbrain\utils\autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)
C:\Users\UsEr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\speechbrain\utils\parameter_transfer.py:234: UserWarning: Requested Pretrainer collection using symlinks on Windows. This might not work; see `LocalStrategy` documentation. Consider unsetting `collect_in` in Pretrainer to avoid symlinking altogether.
  warnings.warn(


In [10]:
import torch

In [11]:
pipeline.to(torch.device("cuda")) # This line moves the pipeline to GPU

In [16]:
DEMO_FILE = {'uri': 'blabal', 'audio': 'audio.wav'}
dz = pipeline(DEMO_FILE)

C:\Users\UsEr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyannote\audio\utils\reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(


In [9]:
DEMO_FILE = {'uri': 'blabal', 'audio': 'audio.wav'}
dz = pipeline(DEMO_FILE)

with open("diarization.txt", "w") as text_file:
    text_file.write(str(dz))

C:\Users\UsEr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyannote\audio\utils\reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(


KeyboardInterrupt: 

In [17]:
print(*list(dz.itertracks(yield_label = True))[:10], sep="\n")

(<Segment(56.4441, 57.6928)>, 'A', 'SPEAKER_05')
(<Segment(58.486, 63.6835)>, 'B', 'SPEAKER_05')
(<Segment(64.3247, 66.586)>, 'C', 'SPEAKER_05')
(<Segment(67.4635, 68.1553)>, 'D', 'SPEAKER_05')
(<Segment(69.016, 71.2603)>, 'E', 'SPEAKER_05')
(<Segment(80.7272, 86.0428)>, 'F', 'SPEAKER_05')
(<Segment(91.0378, 92.6916)>, 'G', 'SPEAKER_05')
(<Segment(94.4803, 95.206)>, 'H', 'SPEAKER_13')
(<Segment(95.0372, 95.2735)>, 'I', 'SPEAKER_05')
(<Segment(96.5728, 97.9735)>, 'J', 'SPEAKER_05')


In [2]:
def millisec(timeStr):
  spl = timeStr.split(":")
  s = (int)((int(spl[0]) * 60 * 60 + int(spl[1]) * 60 + float(spl[2]) )* 1000)
  return s

In [3]:
import re

# Define spacermilli here to use it in this cell
spacermilli = 2000  # This matches the value in cell 10

dz = open('diarization.txt').read().splitlines()
dzList = []
for l in dz:
  start, end =  tuple(re.findall('[0-9]+:[0-9]+:[0-9]+\.[0-9]+', string=l))
  start = millisec(start) - spacermilli
  end = millisec(end)  - spacermilli
  lex = not re.findall('SPEAKER_01', string=l)
  dzList.append([start, end, lex])

print(*dzList[:10], sep='\n')

[54444, 55692, True]
[56485, 61683, True]
[62324, 64585, True]
[65463, 66155, True]
[67015, 69260, True]
[78727, 84042, True]
[89037, 90691, True]
[92479, 93205, True]
[93037, 93273, True]
[94572, 95973, True]


# Preparing audio file from the diarization

Attaching audio segements according to the diarization, with a spacer as the delimiter.

In [ ]:
from pydub import AudioSegment
import re

# Create a spacer for audio segments
spacermilli = 2000  # Using the same value as defined in cell 10
spacer = AudioSegment.silent(duration=spacermilli)
sounds = spacer
segments = []

dz = open('diarization.txt').read().splitlines()
for l in dz:
  start, end =  tuple(re.findall('[0-9]+:[0-9]+:[0-9]+\.[0-9]+', string=l))
  start = int(millisec(start)) #milliseconds
  end = int(millisec(end))  #milliseconds

  segments.append(len(sounds))
  # Load the original audio file
  input_audio = AudioSegment.from_wav("audio.wav")
  # Subtract spacermilli to adjust for the spacer added at the beginning
  sounds = sounds.append(input_audio[start-spacermilli:end-spacermilli], crossfade=0)
  sounds = sounds.append(spacer, crossfade=0)

sounds.export("dz.wav", format="wav") #Exports to a wav file in the current path.

Freeing up some memory

In [18]:
del   sounds, DEMO_FILE, pipeline, spacer,  audio, dz, a, newAudio

NameError: name 'audio' is not defined

# Whisper's Transcriptions

Installing Open AI whisper.

**Important:** There is a version conflict with pyannote.audio resulting in an error (see this RP). Our workaround is to first run Pyannote and then whisper. You can safely ignore the error.


In [2]:
pip install git+https://github.com/openai/whisper.git

  Cloning https://github.com/openai/whisper.git to c:\users\user\appdata\local\temp\pip-req-build-bufnlpsp
  Resolved https://github.com/openai/whisper.git to commit 517a43ecd132a2089d85f4ebc044728a71d49f6e
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Note: you may need to restart the kernel to use updated packages.


  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git 'C:\Users\UsEr\AppData\Local\Temp\pip-req-build-bufnlpsp'


Running Open AI whisper on the prepared audio file. [link text](https://) It writes the transcription into a file.

In [ ]:
!whisper dz.wav --language en --model tiny --device cuda


[00:00.000 --> 00:18.560]  Where, is she?
[00:48.560 --> 01:04.300]  The overworld, the biggest sandbox in the universe, is full of epic tales, millions and billions
[01:04.300 --> 01:05.300]  of them.
[01:05.460 --> 01:10.100]  Well guess what, this one is all mine.
[01:18.820 --> 01:24.020]  My name is Steve, and as a child, I yearn for the mines.
[01:28.980 --> 01:30.980]  But it didn't really work out.
[01:30.980 --> 01:33.620]  Go on, get out of here!
[01:34.420 --> 01:36.420]  So I did a terrible thing.
[01:37.700 --> 01:43.620]  I grew up, and just as I expected, it was a massive bummer.
[02:04.020 --> 02:21.460]  I really put myself out there, and it totally blew up in my face.
[02:22.660 --> 02:25.780]  It was the kind of life that made a man stare into his potatoes.
[02:26.820 --> 02:32.020]  And then I remembered, I couldn't give up on my dream, not yet.
[02:33.860 --> 02:34.820]  The mines!
[02:38.020 --> 02:43.460]  So I bought a pickaxe and a sweet helmet, and this time, 

In [3]:
import torch
torch.cuda.is_available()

True

Reading the transcription file.

In [19]:
pip install -U webvtt-py

Note: you may need to restart the kernel to use updated packages.


In [4]:
import webvtt

captions = [[(int)(millisec(caption.start)), (int)(millisec(caption.end)),  caption.text] for caption in webvtt.read('movie.vtt')]
print(*captions[:8], sep='\n')

[0, 18560, 'Where, is she?']
[48560, 64300, 'The overworld, the biggest sandbox in the universe, is full of epic tales, millions and billions']
[64300, 65300, 'of them.']
[65459, 70100, 'Well guess what, this one is all mine.']
[78820, 84020, 'My name is Steve, and as a child, I yearn for the mines.']
[88980, 90980, "But it didn't really work out."]
[90980, 93620, 'Go on, get out of here!']
[94420, 96420, 'So I did a terrible thing.']


# Matching the Transcriptions and the Diarizations

Matching each trainscrition line to some diarizations, and generating the HTML file. To get the correct timing, we should take care of the parts in original audio that were in no diarization segment.

In [21]:
preS = '<!DOCTYPE html>\n<html lang="en">\n  <head>\n    <meta charset="UTF-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <meta http-equiv="X-UA-Compatible" content="ie=edge">\n    <title>Lexicap</title>\n    <style>\n        body {\n            font-family: sans-serif;\n            font-size: 18px;\n            color: #111;\n            padding: 0 0 1em 0;\n        }\n        .l {\n          color: #050;\n        }\n        .s {\n            display: inline-block;\n        }\n        .e {\n            display: inline-block;\n        }\n        .t {\n            display: inline-block;\n        }\n        #player {\n\t\tposition: sticky;\n\t\ttop: 20px;\n\t\tfloat: right;\n\t}\n    </style>\n  </head>\n  <body>\n    <h2>Yann LeCun: Dark Matter of Intelligence and Self-Supervised Learning | Lex Fridman Podcast #258</h2>\n  <div  id="player"></div>\n    <script>\n      var tag = document.createElement(\'script\');\n      tag.src = "https://www.youtube.com/iframe_api";\n      var firstScriptTag = document.getElementsByTagName(\'script\')[0];\n      firstScriptTag.parentNode.insertBefore(tag, firstScriptTag);\n      var player;\n      function onYouTubeIframeAPIReady() {\n        player = new YT.Player(\'player\', {\n          height: \'210\',\n          width: \'340\',\n          videoId: \'SGzMElJ11Cc\',\n        });\n      }\n      function setCurrentTime(timepoint) {\n        player.seekTo(timepoint);\n   player.playVideo();\n   }\n    </script><br>\n'
postS = '\t</body>\n</html>'

In [ ]:
from datetime import timedelta

html = list(preS)

for i in range(len(segments)):
  idx = 0
  for idx in range(len(captions)):
    if captions[idx][0] >= (segments[i] - spacermilli):
      break;

  while (idx < (len(captions))) and ((i == len(segments) - 1) or (captions[idx][1] < segments[i+1])):
    c = captions[idx]

    start = dzList[i][0] + (c[0] -segments[i])

    if start < 0:
      start = 0
    idx += 1

    start = start / 1000.0
    startStr = '{0:02d}:{1:02d}:{2:02.2f}'.format((int)(start // 3600),
                                            (int)(start % 3600 // 60),
                                            start % 60)

    html.append('\t\t\t<div class="c">\n')
    html.append(f'\t\t\t\t<a class="l" href="#{startStr}" id="{startStr}">link</a> |\n')
    html.append(f'\t\t\t\t<div class="s"><a href="javascript:void(0);" onclick=setCurrentTime({int(start)})>{startStr}</a></div>\n')
    html.append(f'\t\t\t\t<div class="t">{"[Lex]" if dzList[i][2] else "[Yann]"} {c[2]}</div>\n')
    html.append('\t\t\t</div>\n\n')

html.append(postS)
s = "".join(html)

with open("lexicap.html", "w") as text_file:
    text_file.write(s)
print(s)

AttributeError: 'bool' object has no attribute 'split'

: 

In [2]:
# Modify this part where you parse the diarization file
import re

# Define spacermilli here to use it in this cell
spacermilli = 2000  # This matches the value in cell 10

dz = open('diarization.txt').read().splitlines()
dzList = []
for l in dz:
  start, end = tuple(re.findall('[0-9]+:[0-9]+:[0-9]+\\.[0-9]+', string=l))
  start = millisec(start) - spacermilli
  end = millisec(end) - spacermilli
  
  # Extract the speaker ID directly using regex
  speaker_match = re.search(r'SPEAKER_\d+', l)
  speaker_id = speaker_match.group(0) if speaker_match else "UNKNOWN"
  
  # Store the actual speaker ID instead of a boolean
  dzList.append([start, end, speaker_id])

print(*dzList[:10], sep='\n')

NameError: name 'millisec' is not defined